## Image exploration
### Activate caiman environment with conda-forge (mamba)

In [ ]:
!mamba activate caiman

### Check structure and shape of files

In [ ]:
import h5py
import numpy as np

path = r"/home/abl-dell/Downloads/caiman_dataset/2026-04-20_144321-20260614T141833Z-3-002/2026-04-20_144321/raw/stack_1-A1-GCaMP8M_channel_1_obj_bottom/Cam_long_00000.lux.h5"

def explore_h5(filepath):
    with h5py.File(filepath, "r") as f:

        # ---- top-level structure ----
        print("=== Top-level keys ===")
        f.visititems(lambda name, obj: print(f"  {name}  [{type(obj).__name__}]  "
                                            f"shape={getattr(obj, 'shape', '—')}  "
                                            f"dtype={getattr(obj, 'dtype', '—')}"))

        # ---- root-level attributes (microscope metadata) ----
        print("\n=== Root attributes ===")
        for k, v in f.attrs.items():
            print(f"  {k}: {v}")

        # ---- Data array specifics ----
        if "Data" in f:
            d = f["Data"]
            shape = d.shape
            print(f"\n=== Data array ===")
            print(f"  shape  : {shape}")
            print(f"  dtype  : {d.dtype}")
            print(f"  ndim   : {d.ndim}")

            if d.ndim == 3:
                T, H, W = shape
                print(f"  → interpreted as  T={T}  H={H}  W={W}")
                print(f"  → if this is a z-stack movie: {T} time-points or {T} Z-planes")
            elif d.ndim == 4:
                T, Z, H, W = shape
                print(f"  → interpreted as  T={T}  Z={Z}  H={H}  W={W}")
                print(f"  → {T} time-points  ×  {Z} Z-planes")

            # Data attributes (per-dataset metadata)
            print("\n=== Data attributes ===")
            for k, v in d.attrs.items():
                print(f"  {k}: {v}")

            # Quick intensity sanity check (reads only first frame)
            first = d[0].astype(np.float32)
            print(f"\n=== First frame stats ===")
            print(f"  min={first.min():.1f}  max={first.max():.1f}  "
                f"mean={first.mean():.1f}  nonzero={np.count_nonzero(first)}")
            
explore_h5(path)

=== Top-level keys ===
  Data  [Dataset]  shape=(53, 2048, 2048)  dtype=uint16
  metadata  [Dataset]  shape=()  dtype=object

=== Root attributes ===

=== Data array ===
  shape  : (53, 2048, 2048)
  dtype  : uint16
  ndim   : 3
  → interpreted as  T=53  H=2048  W=2048
  → if this is a z-stack movie: 53 time-points or 53 Z-planes

=== Data attributes ===
  element_size_um: [3.    0.208 0.208]

=== First frame stats ===
  min=101.0  max=24584.0  mean=2365.3  nonzero=4194304


In [3]:
with h5py.File(path, "r") as f:
      import json
      raw = f["metadata"][()]
      meta = json.loads(raw)
      print(json.dumps(meta, indent=2))

{
  "processingInformation": {
    "version": "1.0.0",
    "image_id": "2026-04-20T18:24:09.528Z-f27a651f-e8d9-4daa-8ee8-2fe0095013d6",
    "sources": [
      "Luxendo TruLive3D, Embedded v3.17.3, serial-nr: 40073"
    ],
    "contains_beads": false,
    "time_point": "0",
    "channel": "0",
    "stack": "0",
    "stack_description": "A1-rGECO",
    "objective": "bottom",
    "camera": "long",
    "stack_scan_ids": [
      "oc:default_st:0_ch:0_tp:0_pm:none"
    ],
    "voxel_size_um": {
      "width": 0.208,
      "height": 0.208,
      "depth": 3.0
    },
    "image_size_vx": {
      "width": 2048,
      "height": 2048,
      "depth": 53
    },
    "affine_to_sample": [
      {
        "matrix": [
          [
            1.0,
            0.0,
            0.0
          ],
          [
            0.0,
            1.0,
            0.0
          ],
          [
            0.0,
            0.0,
            1.0
          ]
        ],
        "translation": [
          -1023.5,
          -

In [4]:
with h5py.File(path, "r") as f:
    d = f["Data"]          # shape (4900, 2048, 2048)
    n_planes = 7
    plane_idx = 3          # middle plane (0–6), change as needed

    # 700 time points for a single z-plane
    movie = d[plane_idx::n_planes]   # shape (700, 2048, 2048)
    print(movie.shape)               # → (700, 2048, 2048)

FileNotFoundError: [Errno 2] Unable to synchronously open file (unable to open file: name = '20iv26_142407\2026-04-20_142407\raw\stack_0-A1-rGECO_channel_0_obj_bottom\Cam_long_00000.lux.h5', errno = 2, error message = 'No such file or directory', flags = 0, o_flags = 0)

In [38]:
path = r"C:/Users/sosok/Downloads/2026-04-20_144321-20260616T173200Z-3-002/2026-04-20_144321/raw/stack_1-A1-GCaMP8M_channel_1_obj_bottom/Cam_long_00000.lux-001.h5"

import h5py, json

# def read_n_planes(filepath):
#     with h5py.File(filepath, "r") as fh:
#         # print(list(fh["metadata"].keys()))
#         raw = fh["metadata"][()]
#         print(type(raw))
#         print(raw[-1000: -500])
        # walk the full tree
        # fh.visititems(lambda name, obj: print(name))

def read_n_planes(filepath):
    with h5py.File(filepath, "r") as fh:
        raw = fh["metadata"][()]
        meta = json.loads(raw)
        print(int(meta["metaData"]["stack"]["n"]))
        # print(meta["stack"]["n"])       # should print 7
        
read_n_planes(path)


# read_n_planes("/path/to/Cam_long_00000.lux.h5")

# def read_n_planes(filepath: str) -> int:
#       try:
#           with h5py.File(filepath, "r") as fh:
#               if "metaData" not in fh:
#                   return 1          # silent fallback
#               raw = fh["metaData"][()]
#               meta = json.loads(raw)
#               return int(meta["stack"]["n"])
#       except Exception:
#           return 1
      

7
